In [17]:
import pandas as pd
from scipy.stats import ttest_rel
from datetime import datetime
from scipy.stats import ttest_ind

In [18]:
# Load the datasets
referendum_data = pd.read_csv('../data/processed/cleaned_referenda_results.csv')
sp_data = pd.read_csv('../data/processed/party_strength_left_cleaned.csv')
svp_data = pd.read_csv('../data/processed/party_strength_right_cleaned.csv')
old_age_data = pd.read_csv('../data/processed/share_over_64_cleaned.csv')
youth_data = pd.read_csv('../data/processed/share_under_20_cleaned.csv')

In [19]:
# Filter data and create a copy
filtered_data = referendum_data[(referendum_data['date'] >= '2018-01-01') & (referendum_data['date'] < '2023-06-18')].copy()

# Compute turnouts and store results in a new column
filtered_data['turnout'] = ((filtered_data['yeas'] + filtered_data['nays'] + filtered_data['empty'] + filtered_data['invalid']) / filtered_data['eligible_voters'])

# Define municipalities with I-voting
i_voting_ids = [3213, 3392, 3340, 3297, 3238, 3405, 3215, 3251, 3211, 3233, 3352, 3292,  3201, 3374, 3214, 3202, 3407, 3216]

# Filter the data for I-voting and no I-voting municipalities
i_voting_referendum = filtered_data[filtered_data['entity_id'].isin(i_voting_ids)]
no_i_voting_referendum = filtered_data[~filtered_data['entity_id'].isin(i_voting_ids)]

# Calculate mean and standard deviation for both groups
i_voting_stats_ref = i_voting_referendum['turnout'].agg(['mean', 'std'])
no_i_voting_stats_ref = no_i_voting_referendum['turnout'].agg(['mean', 'std'])

# Calculate the differences
stats_difference_referendum = (i_voting_stats_ref - no_i_voting_stats_ref).abs()

no_i_voting_stats_ref, i_voting_stats_ref, stats_difference_referendum


(mean    0.462631
 std     0.123642
 Name: turnout, dtype: float64,
 mean    0.476630
 std     0.122023
 Name: turnout, dtype: float64,
 mean    0.01400
 std     0.00162
 Name: turnout, dtype: float64)

In [20]:
# Merge the youth and old-age data on BFS_NR
dependency_data = pd.merge(youth_data, old_age_data, on=['BFS_NR', 'GEBIET_NAME'], suffixes=('_youth', '_old_age'))

dependency_data['2019_youth'] = dependency_data['2019_youth'] / 100
dependency_data['2019_old_age'] = dependency_data['2019_old_age'] / 100

# Split the data into I-voting and no I-voting municipalities
i_voting_dependency = dependency_data[dependency_data['BFS_NR'].isin(i_voting_ids)]
no_i_voting_dependency = dependency_data[~dependency_data['BFS_NR'].isin(i_voting_ids)]

# Compute mean and standard deviation for Youth and Old-Age Dependency Ratios for both groups
i_voting_stats_age = i_voting_dependency[['2019_youth', '2019_old_age']].agg(['mean', 'std'])
no_i_voting_stats_age = no_i_voting_dependency[['2019_youth', '2019_old_age']].agg(['mean', 'std'])

# Compute the differences
stats_difference_age = (i_voting_stats_age - no_i_voting_stats_age).abs()

no_i_voting_stats_age, i_voting_stats_age, stats_difference_age

(      2019_youth  2019_old_age
 mean    0.356930      0.304386
 std     0.044586      0.056314,
       2019_youth  2019_old_age
 mean    0.356167      0.308889
 std     0.044479      0.051314,
       2019_youth  2019_old_age
 mean    0.000763      0.004503
 std     0.000107      0.005000)

In [21]:
# Merge the SVP and SP data on BFS_NR
voter_share_data = pd.merge(svp_data, sp_data, on=['BFS_NR', 'GEBIET_NAME'], suffixes=('_SVP', '_SP'))

voter_share_data['2019_SVP'] = voter_share_data['2019_SVP'] / 100
voter_share_data['2019_SP'] = voter_share_data['2019_SP'] / 100

# Split the data into I-voting and no I-voting municipalities
i_voting_voter_share = voter_share_data[voter_share_data['BFS_NR'].isin(i_voting_ids)]
no_i_voting_voter_share = voter_share_data[~voter_share_data['BFS_NR'].isin(i_voting_ids)]

# Calculate mean and standard deviation for SVP and SP Voter Shares for both groups
i_voting_stats_party = i_voting_voter_share[['2019_SVP', '2019_SP']].agg(['mean', 'std'])
no_i_voting_stats_party = no_i_voting_voter_share[['2019_SVP', '2019_SP']].agg(['mean', 'std'])

# Calculate the differences
stats_difference_party = (i_voting_stats_party - no_i_voting_stats_party).abs()

no_i_voting_stats_party, i_voting_stats_party, stats_difference_party

(      2019_SVP   2019_SP
 mean  0.354754  0.097649
 std   0.061193  0.038541,
       2019_SVP   2019_SP
 mean  0.333222  0.110222
 std   0.047527  0.045306,
       2019_SVP   2019_SP
 mean  0.021532  0.012573
 std   0.013666  0.006766)

In [22]:
# Function to perform t-test and evaluate significance
def perform_ttest(group1, group2):
    result = ttest_ind(group1, group2, equal_var=False)  # Assume unequal variances
    p_value = result.pvalue
    significance = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.1 else ""
    return p_value, significance

# Gather the data for average turnout 
i_voting_turnout = filtered_data[filtered_data['entity_id'].isin(i_voting_ids)]['turnout']
no_i_voting_turnout = filtered_data[~filtered_data['entity_id'].isin(i_voting_ids)]['turnout']

# Calculate p-value and significance for average turnout
turnout_p_value, turnout_significance = perform_ttest(i_voting_turnout, no_i_voting_turnout)

turnout_significance


'***'

In [23]:
# Gather the data for Youth Dependency Ratio
i_voting_youth_dependency = i_voting_dependency['2019_youth'] 
no_i_voting_youth_dependency = no_i_voting_dependency['2019_youth']

# Calculate p-value and significance for Youth Dependency Ratio
youth_dependency_p_value, youth_dependency_significance = perform_ttest(i_voting_youth_dependency, no_i_voting_youth_dependency)

youth_dependency_significance


''

In [24]:
# Gather the data for Old-Age Dependency Ratio
i_voting_old_age_dependency = i_voting_dependency['2019_old_age']
no_i_voting_old_age_dependency = no_i_voting_dependency['2019_old_age']

# Calculate p-value and significance for Old-Age Dependency Ratio
old_age_dependency_p_value, old_age_dependency_significance = perform_ttest(i_voting_old_age_dependency, no_i_voting_old_age_dependency)

old_age_dependency_significance


''

In [25]:
# Gather the data for SVP Voter Share
i_voting_svp_share = i_voting_voter_share['2019_SVP']
no_i_voting_svp_share = no_i_voting_voter_share['2019_SVP']

# Calculate p-value and significance for VP Voter Share
svp_share_p_value, svp_share_significance = perform_ttest(i_voting_svp_share, no_i_voting_svp_share)

svp_share_significance


''

In [26]:
# Gather the data for SP Voter Share
i_voting_sp_share = i_voting_voter_share['2019_SP']
no_i_voting_sp_share = no_i_voting_voter_share['2019_SP']

# Calculate p-value and significance for SP Voter Share
sp_share_p_value, sp_share_significance = perform_ttest(i_voting_sp_share, no_i_voting_sp_share)

sp_share_significance


''